In [1]:
import os 
from pathlib import Path 
import pandas as pd 
import numpy as np 

from catboost import CatBoostClassifier

import warnings

In [2]:
warnings.filterwarnings("ignore")

In [3]:
os.chdir("..")

In [4]:
print(os.getcwd())

e:\project_archive\new project


In [5]:
# Load data

data_path = Path("data/processed")
if not data_path.exists():
    raise FileNotFoundError

X_train = pd.read_csv(data_path / "x_train.csv")
y_train = np.load(data_path / "y_train.npy")
print("X_train and y_train loaded successfully")

X_test = pd.read_csv(data_path / "x_test.csv")
y_test = np.load(data_path / "y_test.npy")
print("X_test and y_test loaded successfully")

X_train and y_train loaded successfully
X_test and y_test loaded successfully


In [10]:
import yaml 

config_path = Path("config")
with open(config_path / "model_params.yaml", "r") as f:
    config = yaml.safe_load(f)

print(f"config file loaded")
print(config["CatBoost"])

config file loaded
{'iterations': {'min': 100, 'max': 500}, 'learning_rate': {'min': 0.01, 'max': 0.3, 'log': True}, 'depth': {'min': 4, 'max': 10}, 'l2_leaf_reg': {'min': 1.0, 'max': 10.0}, 'random_strength': {'min': 0.0, 'max': 10.0}, 'bagging_temperature': {'min': 0.0, 'max': 5.0}, 'border_count': {'min': 32, 'max': 255}, 'grow_policy': ['SymmetricTree', 'Depthwise'], 'bootstrap_type': ['Bayesian'], 'subsample': {'min': 0.6, 'max': 1.0}, 'objective': ['MultiClass'], 'eval_metric': ['TotalF1'], 'auto_class_weights': ['Balanced', None], 'verbose': [False]}


In [7]:

def suggest_params(trial, config):

    cfg = config["CatBoost"]

    bootstrap_type = trial.suggest_categorical(
        "bootstrap_type",
        cfg["bootstrap_type"],
    )

    params = {
        "iterations": trial.suggest_int(
            "iterations",
            cfg["iterations"]["min"],
            cfg["iterations"]["max"],
            step=50,
        ),

        "learning_rate": trial.suggest_float(
            "learning_rate",
            cfg["learning_rate"]["min"],
            cfg["learning_rate"]["max"],
            log=cfg["learning_rate"]["log"],
        ),

        "depth": trial.suggest_int(
            "depth",
            cfg["depth"]["min"],
            cfg["depth"]["max"],
        ),

        "l2_leaf_reg": trial.suggest_float(
            "l2_leaf_reg",
            cfg["l2_leaf_reg"]["min"],
            cfg["l2_leaf_reg"]["max"],
        ),

        "random_strength": trial.suggest_float(
            "random_strength",
            cfg["random_strength"]["min"],
            cfg["random_strength"]["max"],
        ),

        "bagging_temperature": trial.suggest_float(
            "bagging_temperature",
            cfg["bagging_temperature"]["min"],
            cfg["bagging_temperature"]["max"],
        ),

        "border_count": trial.suggest_int(
            "border_count",
            cfg["border_count"]["min"],
            cfg["border_count"]["max"],
        ),

        "grow_policy": trial.suggest_categorical(
            "grow_policy",
            cfg["grow_policy"],
        ),

        "bootstrap_type": bootstrap_type,

        "objective": cfg["objective"][0],

        "eval_metric": cfg["eval_metric"][0],

        "auto_class_weights": trial.suggest_categorical(
            "auto_class_weights",
            cfg["auto_class_weights"],
        ),

        "verbose": cfg["verbose"][0],

        "random_seed": 42,
    }

    # 'subsample' is only valid with Bernoulli bootstrap
    if bootstrap_type == "Bernoulli":
        params["subsample"] = trial.suggest_float(
            "subsample",
            cfg["subsample"]["min"],
            cfg["subsample"]["max"],
        )

    return params

In [8]:
import optuna
import wandb
from pprint import pprint

from sklearn.model_selection import StratifiedKFold, cross_val_score

def objective(trial, X, y, config, run):

    params = suggest_params(
        trial=trial,
        config=config
    )

    model = CatBoostClassifier(
        **params
        )

    cv_strategy = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=42,
    )

    scores = cross_val_score(
        estimator=model,
        X=X,
        y=y,
        cv=cv_strategy,
        scoring="f1_macro",
        n_jobs=-1,
    )

    score = scores.mean()

    run.log(
        {
            "trial": trial.number,
            "f1_macro_avg": score,
        }
    )

    return score

In [ ]:
# train.py
from sklearn.metrics import f1_score

with wandb.init(
        project="student-drop-enroll-grad-preds",
        name="CatBoost-Optuna-v1",
        group="CatBoost",
        tags=["CatBoost", "Optuna"]
    ) as run:
     
      study = optuna.create_study(
            direction='maximize'
      )

      study.optimize(
            lambda trial: objective(
                  trial,
                  X_train,
                  y_train,
                  config=config,
                  run=run
            ),
            n_trials=50
      )

      best_params = study.best_params

      model = CatBoostClassifier(
            **best_params
      )

      model.fit(X_train, y_train)

      pred = model.predict(X_test)

      f1 = f1_score(
            y_test,
            pred,
            average='macro'
      )

      run.summary["best_cv_score"] = study.best_value
      run.summary["test_f1_macro"] = f1
      run.summary["best_params"] = best_params



[I 2026-06-26 14:41:08,432] A new study created in memory with name: no-name-e99f786b-28b3-4b5b-ab7e-d7d9649cf4c6
[I 2026-06-26 14:41:19,120] Trial 0 finished with value: 0.6244382426960785 and parameters: {'bootstrap_type': 'Bayesian', 'iterations': 150, 'learning_rate': 0.03005074862843646, 'depth': 6, 'l2_leaf_reg': 8.553478148118476, 'random_strength': 7.052770947550247, 'bagging_temperature': 3.8506118140636603, 'border_count': 165, 'grow_policy': 'Depthwise', 'auto_class_weights': None}. Best is trial 0 with value: 0.6244382426960785.
[I 2026-06-26 14:41:29,845] Trial 1 finished with value: 0.6529097544259067 and parameters: {'bootstrap_type': 'Bayesian', 'iterations': 500, 'learning_rate': 0.018000066747860265, 'depth': 7, 'l2_leaf_reg': 6.958628434944174, 'random_strength': 1.1975663361002453, 'bagging_temperature': 4.227588387486111, 'border_count': 89, 'grow_policy': 'SymmetricTree', 'auto_class_weights': None}. Best is trial 1 with value: 0.6529097544259067.
[I 2026-06-26 14

f1_macro_avg,▃▇▁▇▆▇▇▇▇▇▇█▆▇▇▇▇▇▅█▇█▅▂▇██▇█▇▇▆▇▇█████▇
trial,▁▁▁▁▁▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▅▆▆▆▆▆▆▇▇████
f1_macro_avg,0.71524
trial,61


KeyboardInterrupt: 

In [12]:
study.best_value

0.7278704552225732

In [13]:
save_result_path = Path("artifacts/best_params")

save_result_path.mkdir(parents=True, exist_ok=True)

result = {
    "model_name": "CatBoost",
    "best_trial": study.best_trial.number,
    "best_params": study.best_params,
    "best_value": float(study.best_value)
}

with open(save_result_path / "catboost.yaml", "w") as f:
    yaml.safe_dump(result, f, sort_keys=False)
